This notebook contains the implementation of:

1.   **Hourglass Persistence**(Definition 9) on a graph.
2.   A (modification of) **(f,g)-FB Persistence** on a graph. We also implement Algorithm 3 to obtain (a modification of) Backward Persistence.

In the second part, we also verify the examples in Figure 3(c),(d) in the paper as well. We do note that the actual paper verified these two examples using theoretical arguments.

The implementation in this notebook uses the approach outlined in Section 5.1. The code in this notebook is not used in the experimental validations of our paper. We only included this for completeness. The code here may be optimized.

In [ ]:
# Uncomment if needed to install the gudhi package
# !pip install gudhi

import torch
import itertools
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import random
import gudhi
import math

I. Implementation of Hourglass Persistence.

In [ ]:
# Given a filtration function f on a graph G, viewed as a list of values on
# its vertices and edges, return the intermediate complexes associated to this.
def get_intermediate_complexes(G, vertex_val, edge_val):
  value_list = list(set(vertex_val + edge_val))
  value_list.sort()
  vertices, edges = G

  intermediate_complex_list = []
  for val in value_list:
    ic = (val, [], [])
    for i in range(0, len(vertex_val)):
      if vertex_val[i] == val:
        ic[1].append(i)
    for j in range(0, len(edge_val)):
      if edge_val[j] == val:
        ic[2].append(j)
        v, w = edges[j]
        if v not in ic[1]:
          ic[1].append(v)
        if w not in ic[1]:
          ic[1].append(w)
    intermediate_complex_list.append(ic)

  return intermediate_complex_list

# Given a graph G and an order of inclusions and contractions (called hourglass_list)
# and the list of intermediate complexes.
# Returns the hourglass persistence associated to this input.

def hourglass_persistence(G, hourglass_list, intermediate_complex_list):
  st = gudhi.SimplexTree()
  # Record list of intermediate complexes seen
  witness_list = []
  # Record list of vertices and edges appearing and disappearing
  inclusion_list = [[], []]
  contraction_list = [[], []]

  vertices, edges = G

  # Because gudhi's simplex tree cannot take a vertex labeled with a negative
  # number, for consistency here we are going to take this to be a number >
  # the number of vertices of G
  initial_node = len(vertices) + 1

  # print(vertices)
  # print(edges)

  # Adding the initial vertex at t = -1
  st.insert([initial_node], filtration=-1)

  for val in range(0, len(hourglass_list)):
    # print(val)
    i = hourglass_list[val]
    if i not in witness_list:
      # This means we are doing inclusion
      witness_list.append(i)
      _, ic_v, ic_e = intermediate_complex_list[i]

      # Insert vertices
      for current_v in ic_v:
        if current_v not in inclusion_list[0]:
          st.insert([current_v], filtration=val)
          inclusion_list[0].append(current_v)

      # Inserting edges
      for e_indx in ic_e:
        current_e = edges[e_indx]
        if current_e not in inclusion_list[1]:
          st.insert(current_e, filtration=val)
          inclusion_list[1].append(current_e)

    else:
      # This means we are doing contraction
      _, ic_v, ic_e = intermediate_complex_list[i]
      for current_v in ic_v:
        if current_v not in contraction_list[0]:
          st.insert([initial_node, current_v], filtration=val)
          contraction_list[0].append(current_v)
      for e_indx in ic_e:
        current_e = edges[e_indx]
        if current_e not in contraction_list[1]:
          st.insert([initial_node, current_e[0], current_e[1]], filtration=val)
          contraction_list[1].append(current_e)

    # print(inclusion_list)
    # print(contraction_list)
    # print("------------------")
  st.make_filtration_non_decreasing()
  dgms = st.persistence(min_persistence=-1, persistence_dim_max=True)

  hg_persistence = []
  # Clear trivial births in cycles. This loop also removes s(-1.0, inf)
  for dim, bd in dgms:
    if bd[0] == -1:
      # This is the initial node
      continue
    if dim != 1 or bd[0] != bd[1]:
      hg_persistence.append((dim, [bd[0], bd[1]]))

  # Mark only one of the earliest born vertices to die at inf
  for dim, bd in hg_persistence:
    if bd[0] == 0:
      # This is one of the earliest born vertices:
      bd[1] = "inf"
      break

  return hg_persistence

In [ ]:
# Example: G = 3-cycle
G = [[0, 1, 2], [(0,1), (1,2), (0,2)]]
ic_list = get_intermediate_complexes(G, [0, 0, 1], [0, 1, 1])
print(ic_list)
output = hourglass_persistence(G, [1, 0, 0, 1], ic_list)
print("Hourglass Persistence: ", output)
print("----------------")

# Example: Contraction Makes a Cycle
G = [[0, 1, 2], [(0,1), (1,2)]]
ic_list = get_intermediate_complexes(G, [0, 1, 0], [5.5, 6.5])
print(ic_list)
output = hourglass_persistence(G, [0, 1, 0, 2, 3, 2, 3, 0], ic_list)
print("Hourglass Persistence: ", output)

[(0, [0, 1], [0]), (1, [2, 1, 0], [1, 2])]
Hourglass Persistence:  [(1, [1.0, 3.0]), (0, [0.0, 'inf']), (0, [0.0, 0.0]), (0, [0.0, 0.0])]
----------------
[(0, [0, 2], []), (1, [1], []), (5.5, [0, 1], [0]), (6.5, [1, 2], [1])]
Hourglass Persistence:  [(1, [4.0, 6.0]), (0, [0.0, 'inf']), (0, [0.0, 2.0]), (0, [1.0, 3.0])]


II. Implementation of (f,g)-FB Persistence.

Here we explain the slight modification. In the proof of Proposition 7, we noted that the time of when to place the point v_+ may affect the 0-th dimensional diagram. But for (f,g)-FB persistence, placing v_+ after inclusions and before contractions will not affect the birth/death times of other vertices.

The latter is the approach taken in the algorithms below. We did not remove the extraneous point out of the final persistence diagram, but this will not affect expressivity.

In [ ]:
def return_max_val_of_fil(f):
  num_edges = len(f[1])
  f_max = max(map(max, f)) if num_edges > 0 else max(f[0])
  return f_max

# Modifies g so that the values of g appears later than that of f
def modify_g(f, g, start):
  g_new_v = []
  g_new_e = []

  for v in g[0]:
    g_new_v.append(v + start)
  for e in g[1]:
    g_new_e.append(e + start)

  return [g_new_v, g_new_e]

# Computes the (f,g)-FB persistence
# start specifies where to start g after f.
def compute_fg_persistence(G, f, g, start):
  st = gudhi.SimplexTree()
  f_max = return_max_val_of_fil(f)
  g_new = modify_g(f, g, start)
  initial_node = 0

  # Adding the initial vertex at t > f_max but before doing g
  st.insert([initial_node], filtration=f_max + 0.5)

  # Adding vertices
  for i in range(0, len(G[0])):
    current_v = G[0][i]
    v_val = f[0][i]
    st.insert([current_v], filtration=v_val)

  # Adding edges
  for i in range(0, len(G[1])):
    current_e = G[1][i]
    e_val = f[1][i]
    st.insert(current_e, filtration=e_val)

  # Adding triangles
  for i in range(0, len(G[0])):
    current_v = G[0][i]
    v_val = g_new[0][i]
    st.insert([initial_node, current_v], filtration=v_val)

  for i in range(0, len(G[1])):
    current_e = G[1][i]
    e_val = g_new[1][i]
    st.insert([initial_node, current_e[0], current_e[1]], filtration=e_val)


  st.make_filtration_non_decreasing()
  dgms = st.persistence(min_persistence=-1, persistence_dim_max=True)

  return dgms

In [ ]:
# We can also single out the f and g parts as follows:

# Compute F persistence
def compute_fp(G, f):
  st = gudhi.SimplexTree()
  for i in range(0, len(G[0])):
    current_v = G[0][i]
    v_val = f[0][i]
    st.insert([current_v], filtration=v_val)

  # Adding edges
  for i in range(0, len(G[1])):
    current_e = G[1][i]
    e_val = f[1][i]
    st.insert(current_e, filtration=e_val)

  st.make_filtration_non_decreasing()
  dgms = st.persistence(min_persistence=-1, persistence_dim_max=True)
  return dgms


# Computes the persistence of a sequence of subgraphs specified by the
# filtration g.
def compute_ch(G, g):
  st = gudhi.SimplexTree()
  g_new = g
  initial_node = 0

  # print(f)
  # Adding the initial vertex at t = -1
  st.insert([initial_node], filtration=-1)

  # Adding vertices
  for i in range(0, len(G[0])):
    current_v = G[0][i]
    v_val = 0
    st.insert([current_v], filtration=v_val)

  # Adding edges
  for i in range(0, len(G[1])):
    current_e = G[1][i]
    e_val = 0
    st.insert(current_e, filtration=e_val)

  # Adding triangles
  for i in range(0, len(G[0])):
    current_v = G[0][i]
    v_val = g_new[0][i]
    st.insert([initial_node, current_v], filtration=v_val)

  for i in range(0, len(G[1])):
    current_e = G[1][i]
    e_val = g_new[1][i]
    st.insert([initial_node, current_e[0], current_e[1]], filtration=e_val)


  st.make_filtration_non_decreasing()
  dgms = st.persistence(min_persistence=-1, persistence_dim_max=True)

  return dgms

We may realize Backward Persistence of f as applying compute_ch to f^b, where f^b is the filtration specified by ALgorithm 3 in the paper.

In [ ]:
# The following is an implementation of Algorithm 3 in the paper.

# Given a vertex-based filtration function f, produces the function f^b specifying
# its backward persistence.
def filtration_to_backward(G, f):
  vertex_value = [-1 for _ in range(0, len(G[0]))]
  for i in range(0, len(G[1])):
    # print(G)
    edge = G[1][i]
    e_val = f[1][i]
    v = edge[0]
    w = edge[1]

    # if vertex value has been assigned
    if vertex_value[v-1] != -1:
      # Only increase value if the edge instantiated is higher value
      if e_val > vertex_value[v-1]:
        vertex_value[v-1] = e_val
    else:
      vertex_value[v-1] = e_val

  # If the vertex is isolated, assign it its value here
  for i in range(0, len(vertex_value)):
    if vertex_value[i] == -1:
      vertex_value[i] = f[0][i]

  v_max = max(vertex_value)

  return v_max, vertex_value

Here we write some utility functions for running the methods above.

In [ ]:
# Convert a networkx graph to the input datatype for the code
def networkxg_to_input(NG):
  # Shift the index up by 1 to avoid having a vertex labeled 0
  nodes_list = list(map(lambda x: x + 1, NG.nodes))
  edge_list = list(map(lambda x: (x[0] + 1, x[1] + 1), NG.edges))
  return [nodes_list, edge_list]

# By default, a graph's vertex index should start at 0, rather than 1.
# Also, by default, for simplicity here a graph is finite simple (but of course it could have multi-edges and self-loops in contraction)

# Given a pair of graphs TG1, TG2, both networkX graphs
# Outputs pairs (G1, f1, g1), (G2, f2, g2) where:
# f1 and f2 are the filtrations of G1 and G2 in the forward direction.
# g1 and g2 are the filtrations of G1 and G2 in the contraction direction.

# Here the choices of f1, g1, f2, g2 depend on which part is commented out in
# the function.
def dataset_entry_to_input(TG1, TG2):

  G1 = networkxg_to_input(TG1)
  G2 = networkxg_to_input(TG2)

  # Betweenness
  # betweenness1 = list(map(lambda x: round(x, 3), nx.betweenness_centrality(TG1).values()))
  # betweenness2 = list(map(lambda x: round(x, 3), nx.betweenness_centrality(TG2).values()))
  # f1 = [betweenness1, list(map(lambda x: max(betweenness1[x[0]], betweenness1[x[1]]), TG1.edges))]
  # f2 = [betweenness2, list(map(lambda x: max(betweenness2[x[0]], betweenness2[x[1]]), TG2.edges))]

  # Closeness
  # closeness1 = list(map(lambda x: round(x, 3), nx.closeness_centrality(TG1).values()))
  # closeness2 = list(map(lambda x: round(x, 3), nx.closeness_centrality(TG2).values()))
  # h1 = [closeness1, list(map(lambda x: max(closeness1[x[0]], closeness1[x[1]]), TG1.edges))]
  # h2 = [closeness2, list(map(lambda x: max(closeness2[x[0]], closeness2[x[1]]), TG2.edges))]

  # Degree
  v_val = list(map(lambda x: random.randint(1, 10), TG1.nodes))
  e_val = list(map(lambda x: max(v_val[x[0]], v_val[x[1]]), TG1.edges))
  f1 = [v_val, e_val]
  f2 = [v_val, e_val]

  # Here we choose to make g1, g2 to be produced using Algorithm 3
  vmax1, vval1 = filtration_to_backward(G1, f1)
  vmax2, vval2 = filtration_to_backward(G2, f2)

  v_max = max(vmax1, vmax2)
  v_output1 = list(map(lambda x: round(-x + v_max + 1, 3), vval1))
  v_output2 = list(map(lambda x: round(-x + v_max + 1, 3), vval2))

  e_output1 = []
  e_output2 = []
  for i in range(0, len(G1[1])):
    edge = G1[1][i]
    e_val1 = f1[1][i]
    e_output1.append(round(-e_val1 + v_max + 1, 3))

  for i in range(0, len(G2[1])):
    edge = G2[1][i]
    e_val2 = f2[1][i]
    e_output2.append(round(-e_val2 + v_max + 1, 3))

  g1 = [v_output1, e_output1]
  g2 = [v_output2, e_output2]

  return (G1, f1, g1), (G2, f2, g2)

def pd_to_multiset(pd_list):
  output = {}
  for item in pd_list:
    output[str(item)] = 0
  for item in pd_list:
    output[str(item)] += 1
  return output

def check_pd_equal(pd1, pd2):
  pd1_list = pd_to_multiset(pd1)
  pd2_list = pd_to_multiset(pd2)

  return pd1_list, pd2_list, pd1_list == pd2_list

In [ ]:
# Here is an example for how to run this:

NG = nx.from_edgelist([(0, 1), (1, 2), (2, 3), (1, 4), (4, 5)])
NH = nx.from_edgelist([(0, 1), (1, 2), (1, 3), (3, 4), (4, 5)])

G = networkxg_to_input(NG)
H = networkxg_to_input(NH)

entry = dataset_entry_to_input(NG, NH)
G1, f1, g1 = entry[0]
G2, f2, g2 = entry[1]

start = max(return_max_val_of_fil(f1), return_max_val_of_fil(f2))
A = compute_fg_persistence(G1, f1, g1, start)
B = compute_fg_persistence(G2, f2, g2, start)

A_clear, B_clear, result = check_pd_equal(A, B)
print(result)

False


Now we evaluate our methods on Figure 3(c) and 3(d).

In [ ]:
# Testing this on the Example of Figure 3(c) in Paper

# FB can differentiate
G = [[1, 2, 3, 4], [[1,2], [2,3], [2,4]]]
H = [[1,2,3,4], [[1,2], [2,3], [3,4]]]

f1 = [[1, 1, 2, 2], [1, 2, 2]]
f2 = [[2,1,1,2], [2,1,2]]

vmax1, vval1 = filtration_to_backward(G, f1)
vmax2, vval2 = filtration_to_backward(H, f2)

v_max = max(vmax1, vmax2)
v_output1 = list(map(lambda x: round(-x + v_max + 1, 3), vval1))
v_output2 = list(map(lambda x: round(-x + v_max + 1, 3), vval2))

e_output1 = []
e_output2 = []
for i in range(0, len(G[1])):
  edge = G1[1][i]
  e_val1 = f1[1][i]
  e_output1.append(round(-e_val1 + v_max + 1, 3))

for i in range(0, len(H[1])):
  edge = G2[1][i]
  e_val2 = f2[1][i]
  e_output2.append(round(-e_val2 + v_max + 1, 3))

g1 = [v_output1, e_output1]
g2 = [v_output2, e_output2]

A = compute_fg_persistence(G, f1, g1, 3)
B = compute_fg_persistence(H, f2, g2, 3)

A_clear, B_clear, result = check_pd_equal(A, B)
print(result)

False


In [ ]:
# Testing this on the Example of Figure 3(d) in Paper

# FB cannot differentiate
G = [[1, 2, 3, 4, 5, 6], [[1,2], [3,4], [2,5], [4, 5], [5,6]]]
H = [[1,2,3,4,5,6], [[1,3], [2,3], [3,4], [4,5], [5,6]]]

f1 = [[1, 2, 1, 2, 3, 1], [2, 2, 3, 3, 3]]
f2 = [[1,1,3,2,2,1], [3,3,3,2,2]]

vmax1, vval1 = filtration_to_backward(G, f1)
vmax2, vval2 = filtration_to_backward(H, f2)

v_max = max(vmax1, vmax2)
v_output1 = list(map(lambda x: round(-x + v_max + 1, 3), vval1))
v_output2 = list(map(lambda x: round(-x + v_max + 1, 3), vval2))

e_output1 = []
e_output2 = []
for i in range(0, len(G[1])):
  edge = G1[1][i]
  e_val1 = f1[1][i]
  e_output1.append(round(-e_val1 + v_max + 1, 3))

for i in range(0, len(H[1])):
  edge = G2[1][i]
  e_val2 = f2[1][i]
  e_output2.append(round(-e_val2 + v_max + 1, 3))

g1 = [v_output1, e_output1]
g2 = [v_output2, e_output2]

A = compute_fg_persistence(G, f1, g1, 3)
B = compute_fg_persistence(H, f2, g2, 3)

A_clear, B_clear, result = check_pd_equal(A, B)
print(result)

True
